# ARPG-L Baseline Reproduction (Colab)

**Before running:** Go to **Runtime > Change runtime type** and select **A100 GPU** (or T4/V100 if A100 is unavailable).

This notebook:
1. Verifies GPU access
2. Clones the repo & installs deps
3. Downloads model weights + reference batch
4. Runs a 1k smoke test
5. Runs the full 50k-sample baseline
6. Computes FID (target: ~2.38)

## 1. Verify GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > A100 GPU")

## 2. Clone repo & install dependencies

**Option A** - Clone from GitHub (edit the URL below):  
**Option B** - If you uploaded the repo to Google Drive, mount it instead (see commented cell).

In [ ]:
# Option A: Clone from GitHub (replace with your repo URL)
!git clone https://github.com/YOUR_USERNAME/ARPG.git /content/ARPG
%cd /content/ARPG

In [ ]:
# Option B: Mount Google Drive (uncomment if you uploaded the repo there)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/ARPG-main /content/ARPG
# %cd /content/ARPG

In [ ]:
# Install runtime deps (PyTorch is already in Colab, so we just need the extras)
!pip install -q transformers einops Pillow tqdm numpy

In [ ]:
# Verify the repo imports work
import sys, subprocess
result = subprocess.run(
    [sys.executable, "-c",
     "from models.arpg import ARPG_models; print('Available models:', sorted(ARPG_models.keys()))"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)
    raise RuntimeError("Model import failed")

## 3. Download assets

Downloads ~2.5 GB total:
- `weights/arpg_300m.pt` (ARPG-L checkpoint)
- `weights/vq_ds16_c2i.pt` (LlamaGen VQ tokenizer)
- `eval/VIRTUAL_imagenet256_labeled.npz` (ImageNet-256 reference batch)
- `external/guided-diffusion` (OpenAI FID evaluator)

In [ ]:
!bash scripts/download_baseline_assets.sh

In [ ]:
# Verify all assets are present
import os
assets = [
    "weights/arpg_300m.pt",
    "weights/vq_ds16_c2i.pt",
    "eval/VIRTUAL_imagenet256_labeled.npz",
    "external/guided-diffusion/evaluations/evaluator.py",
]
for path in assets:
    exists = os.path.exists(path)
    size = f"{os.path.getsize(path)/1e6:.1f} MB" if exists else "MISSING"
    status = "OK" if exists else "FAIL"
    print(f"  [{status}] {path}  ({size})")

## 4. Smoke test (1k samples)

Quick sanity check. Should take ~5-10 min on a T4, ~2-3 min on an A100.  
Uses `--no-compile` to avoid potential `torch.compile` issues on Colab.

In [ ]:
!ARPG_NO_COMPILE=1 bash scripts/run_arpgl_smoke.sh

In [ ]:
# Check that samples were generated
import glob
smoke_dir = "samples/arpgl-baseline"
sample_dirs = glob.glob(f"{smoke_dir}/ARPG-L-*")
if sample_dirs:
    pngs = glob.glob(f"{sample_dirs[0]}/*.png")
    print(f"Generated {len(pngs)} sample images in {sample_dirs[0]}")
    npzs = glob.glob(f"{smoke_dir}/*.npz")
    if npzs:
        print(f"NPZ file: {npzs[0]}")
else:
    print("ERROR: No sample directory found!")

In [ ]:
# Preview a few generated samples
from PIL import Image
import matplotlib.pyplot as plt

if sample_dirs:
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for i, ax in enumerate(axes):
        img = Image.open(f"{sample_dirs[0]}/{i:06d}.png")
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f"Sample {i}")
    plt.suptitle("Smoke test samples")
    plt.tight_layout()
    plt.show()

## 5. Full 50k-sample baseline

This is the real run. Time estimates (1 GPU):
- **A100**: ~30-45 min
- **V100**: ~1-2 hours
- **T4**: ~2-4 hours

**Important:** Before running, delete the smoke test output so the full run starts clean.

In [ ]:
# Clean up smoke test output
!rm -rf samples/arpgl-baseline

In [ ]:
!ARPG_NO_COMPILE=1 bash scripts/run_arpgl_full.sh

In [ ]:
# Verify 50k samples + NPZ
sample_dirs = glob.glob(f"{smoke_dir}/ARPG-L-*")
if sample_dirs:
    pngs = glob.glob(f"{sample_dirs[0]}/*.png")
    print(f"Generated {len(pngs)} sample images")
npzs = glob.glob(f"{smoke_dir}/*.npz")
if npzs:
    import numpy as np
    data = np.load(npzs[0])
    print(f"NPZ shape: {data['arr_0'].shape}")
    print(f"NPZ path: {npzs[0]}")

## 6. Compute FID

Uses OpenAI's `guided-diffusion` evaluator. Target FID: **~2.38** (acceptable: 2.2-2.6).

In [ ]:
# Install eval deps (scipy and tensorflow for the evaluator)
!pip install -q scipy tensorflow

In [ ]:
!bash scripts/eval_arpgl_baseline.sh

## 7. Save results to Drive (optional)

Save the NPZ and FID output so you don't lose them if the Colab session disconnects.

In [ ]:
# Uncomment to save results to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/ARPG-results
# !cp samples/arpgl-baseline/*.npz /content/drive/MyDrive/ARPG-results/
# print("Results saved to Google Drive")